# 14. Occlusion before and after OAF (CPU only, ~3 min)

Did OAF reduce the identity splits that are **caused by occlusion**? Same split definition as the diagnosis (notebook 04): cut at 90% pair precision, same-person pairs on different cameras, each person weighted equally. Pairs are grouped by how hidden the person is (none / some / heavy / rarely visible).

**Needs:** the diagnosis measurements (notebook 04) and the fingerprints from notebooks 05–07.

**Paper:** Table 5.

In [ ]:
# Did OAF reduce the splits CAUSED BY OCCLUSION? (standalone, CPU, ~3 min)
# Same split definition as the diagnosis (Table 3): cut at 90% pair precision,
# pairs of the same person on different cameras, each person weighted equally.
import os, glob, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])

RES = os.path.dirname(os.path.dirname(find([f'{SD}/Data/MEVID-results/LookFactors-SameClothes/clip_measurements.csv',
                                            f'{SD}/**/LookFactors-SameClothes/clip_measurements.csv'])))
LF, CS, OA, TM = (f'{RES}/{d}' for d in ['LookFactors-SameClothes', 'CSCI-SameClothes',
                                         'OcclusionAware-SameClothes', 'ThreeModels-SameClothes'])
SC_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
                               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
info = np.loadtxt(f'{SC_DIR}/annotations/track_test_info.txt').astype(np.int64)
pid, cam = info[:, 2], info[:, 4]
CL = pd.read_csv(f'{LF}/clip_measurements.csv', index_col=0)
assert len(CL) == len(pid) and (CL.pid.to_numpy() == pid).all(), 'clip order differs'

unit = lambda x: x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
FEATS = {}
for v, fn in [('all frames', f'{CS}/test_templates_csci_sc.npz'),
              ('ours (OAF)', f'{OA}/test_templates_csci_visible.npz'),
              ('control: random', f'{OA}/test_templates_csci_random.npz'),
              ('control: feature', f'{OA}/test_templates_csci_feature.npz')]:
    z = np.load(fn); assert (z['pid'] == pid).all()
    FEATS[('CSCI', v)] = unit(z['templates'].astype(np.float32))
for m in ['FastReID', 'OSNet']:
    fn = f'{TM}/fingerprints_{m}.npz'
    if os.path.exists(fn):
        z = np.load(fn)
        FEATS[(m, 'all frames')] = unit(z['all frames'])
        FEATS[(m, 'ours (OAF)')] = unit(z['ours (pose: person visible)'])
print(len(FEATS), 'fingerprint sets loaded')

# same-person, cross-camera pairs and their occlusion level (as in Table 1)
iu, ju = np.triu_indices(len(pid), 1)
k = (pid[iu] == pid[ju]) & (cam[iu] != cam[ju])
I, J = iu[k], ju[k]
occ = np.maximum(CL.occluded.to_numpy()[I], CL.occluded.to_numpy()[J])
OCC = np.where(occ < 0.25, 'none', np.where(occ < 0.5, 'some', 'heavy'))
vi, vj = CL.view.to_numpy()[I], CL.view.to_numpy()[J]
UNK = (vi == 'unknown') | (vj == 'unknown')          # mostly clips with no visible person
_, g = np.unique(pid[I], return_inverse=True)
W = 1.0 / np.bincount(g)[g]                            # each person counts once
print(f'{len(I):,} pairs | occlusion none/some/heavy: '
      + ' / '.join(str((OCC == x).sum()) for x in ['none', 'some', 'heavy']))

def pair_precision(lab, p):
    M = pd.crosstab(p, lab).to_numpy().astype(float); nc = M.sum(0)
    same = (nc * (nc - 1)).sum()
    return (M * (M - 1)).sum() / same if same else 1.0

def ids_at_precision(X, target=0.90):                  # same rule as the diagnosis
    D = np.clip(1 - X @ X.T, 0, None); D = (D + D.T) / 2; np.fill_diagonal(D, 0)
    L = linkage(squareform(D, checks=False), 'average')
    best = fcluster(L, 1e-9, 'distance')
    for t in np.linspace(0.01, 1.5, 150):
        lab = fcluster(L, t, 'distance')
        if pair_precision(lab, pid) >= target:
            best = lab
        else:
            break
    return best

rate = lambda s, m: round(100 * np.average(s[m], weights=W[m]), 1)
ROWS = []
for (model, v), X in tqdm(FEATS.items(), desc='models x variants'):
    lab = ids_at_precision(X)
    s = (lab[I] != lab[J]).astype(float)
    ROWS.append({'model': model, 'variant': v,
                 'all pairs': rate(s, np.ones_like(s, bool)),
                 'no occlusion': rate(s, OCC == 'none'),
                 'some occlusion': rate(s, OCC == 'some'),
                 'heavy occlusion': rate(s, OCC == 'heavy'),
                 'view unknown (empty clips)': rate(s, UNK),
                 'IDs per person': round(pd.Series(lab).groupby(pid).nunique().mean(), 2)})
T = pd.DataFrame(ROWS).set_index(['model', 'variant'])
print('\nSPLIT RATE % (lower is better), each person weighted equally:')
display(T)
T.to_csv(f'{LF}/split_rate_by_occlusion_before_after.csv')
print('Saved to', f'{LF}/split_rate_by_occlusion_before_after.csv')